# IE4483 — Four Traditional ML Methods: Hyperparameter Tuning

Four sentiment classifiers are evaluated on the shared product-review splits:

1. **TF-IDF + Logistic Regression**
2. **TF-IDF + Linear SVM**
3. **TF-IDF + Multinomial Naive Bayes**
4. **TF-IDF + Truncated SVD + Logistic Regression**

For each method, it **tests different parameter values**, selects the best settings using **3-fold cross-validation on training data only**, and evaluates the selected model **once on the original validation split**.

**Saved output (only five files):** `model_comparison.csv` and **four confusion-matrix PNGs**. No individual prediction CSVs, no separate Naive Bayes tuning CSV, and no separate trial files.

**Metric for choosing the best configuration:** Negative F1 (label `0`), since negative reviews are the minority class.

## 1. Import libraries

Run from the repository root. The notebook reads `data/processed/train_split.json` and `data/processed/val_split.json`.

If required, install the dependencies with `pip install pandas scikit-learn matplotlib`.

In [ ]:
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
    make_scorer,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import Normalizer
from sklearn.svm import LinearSVC

## 2. Experiment settings

- **`QUICK_MODE = True` (recommended first run):** tests a focused set of parameters, including `C = 0.1, 1, 10` for the SVD model.
- **`QUICK_MODE = False`:** additionally tries more SVD dimensions, class weights, and unigram versus bigram TF-IDF. This takes longer.
- Keep `RANDOM_SEED` fixed for a fair comparison across models.

In [ ]:
QUICK_MODE = True
RANDOM_SEED = 42
CV_FOLDS = 3
N_JOBS = 2  # Set to 1 to reduce memory use.

DATA_DIR = Path("data/processed")
OUTPUT_DIR = Path("traditional_ml_4_methods_tuned")

if not (DATA_DIR / "train_split.json").exists():
    raise FileNotFoundError(
        "Cannot find data/processed/train_split.json. "
        "Place this notebook in the extracted project's root folder."
    )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Results folder:", OUTPUT_DIR.resolve())

## 3. Load the same train and validation splits as BERT / RNN

The review texts are used directly. The TF-IDF vectorizer is included **inside each pipeline**, so it is fitted separately for each cross-validation training fold. This prevents data leakage.

In [ ]:
train = pd.read_json(DATA_DIR / "train_split.json")
val = pd.read_json(DATA_DIR / "val_split.json")

X_train = train["reviews"].fillna("").astype(str)
y_train = train["sentiments"].astype(int)
X_val = val["reviews"].fillna("").astype(str)
y_val = val["sentiments"].astype(int)

print(f"Training reviews: {len(X_train):,}")
print(f"Validation reviews: {len(X_val):,}")
print("Training labels:", y_train.value_counts().sort_index().to_dict())
print("Validation labels:", y_val.value_counts().sort_index().to_dict())

## 4. Define the four methods and the parameters to test

For all methods, the initial TF-IDF settings are `ngram_range=(1, 2)`, `min_df=2`, `max_features=20,000`, `sublinear_tf=True`.

- **Logistic Regression / SVM:** test the regularization `C` and class weights.
- **Naive Bayes:** test smoothing `alpha` and whether to learn class priors.
- **SVD + Logistic Regression:** test SVD feature dimensions (`n_components`) **and** Logistic Regression `C`. SVD is followed by `Normalizer`.

SVD can produce negative feature values, so Naive Bayes uses TF-IDF directly rather than the SVD-transformed features.

In [ ]:
def new_tfidf():
    return TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_features=20_000,
        sublinear_tf=True,
    )

# Candidate values for cross-validation.
C_VALUES = [0.1, 1.0, 10.0]
CLASS_WEIGHTS = [None, "balanced"]
NB_ALPHA_VALUES = [0.01, 0.1, 0.5, 1.0, 2.0]
SVD_DIMENSIONS = [100, 200] if QUICK_MODE else [50, 100, 200, 300]
NGRAM_OPTIONS = [(1, 2)] if QUICK_MODE else [(1, 1), (1, 2)]

model_searches = {
    "Logistic Regression": (
        Pipeline([
            ("tfidf", new_tfidf()),
            ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_SEED)),
        ]),
        {
            "tfidf__ngram_range": NGRAM_OPTIONS,
            "clf__C": C_VALUES,
            "clf__class_weight": CLASS_WEIGHTS,
        },
    ),
    "Linear SVM": (
        Pipeline([
            ("tfidf", new_tfidf()),
            ("clf", LinearSVC(max_iter=5000, random_state=RANDOM_SEED)),
        ]),
        {
            "tfidf__ngram_range": NGRAM_OPTIONS,
            "clf__C": C_VALUES,
            "clf__class_weight": CLASS_WEIGHTS,
        },
    ),
    "Naive Bayes": (
        Pipeline([
            ("tfidf", new_tfidf()),
            ("clf", MultinomialNB()),
        ]),
        {
            "tfidf__ngram_range": NGRAM_OPTIONS,
            "clf__alpha": NB_ALPHA_VALUES,
            "clf__fit_prior": [True, False],
        },
    ),
    "SVD + Logistic Regression": (
        Pipeline([
            ("tfidf", new_tfidf()),
            ("svd", TruncatedSVD(n_iter=5, random_state=RANDOM_SEED)),
            ("normalize", Normalizer()),
            ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_SEED)),
        ]),
        {
            "tfidf__ngram_range": NGRAM_OPTIONS,
            "svd__n_components": SVD_DIMENSIONS,
            "clf__C": C_VALUES,
            "clf__class_weight": ["balanced"] if QUICK_MODE else CLASS_WEIGHTS,
        },
    ),
}

for name, (_, params) in model_searches.items():
    n_runs = 1
    for choices in params.values():
        n_runs *= len(choices)
    print(f"{name}: {n_runs} configurations × {CV_FOLDS} folds = {n_runs * CV_FOLDS} CV fits")

## 5. Tune all models, evaluate, and save confusion matrices

The code selects settings using **training-set cross-validation Negative F1**, not the held-out validation scores. It then fits the selected configuration on all training reviews and evaluates once on validation.

The notebook saves one validation confusion matrix per model as a PNG.

In [ ]:
cv = StratifiedKFold(
    n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED
)
scoring = {
    "negative_f1": make_scorer(f1_score, pos_label=0, zero_division=0),
    "macro_f1": "f1_macro",
    "accuracy": "accuracy",
}

rows = []

for model_name, (pipeline, param_grid) in model_searches.items():
    print(f"\n===== {model_name} =====", flush=True)
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring=scoring,
        refit="negative_f1",
        cv=cv,
        n_jobs=1 if model_name.startswith("SVD") else N_JOBS,
        return_train_score=False,
        error_score="raise",
    )

    start = perf_counter()
    search.fit(X_train, y_train)
    search_time = perf_counter() - start
    print("Best CV Negative F1:", round(search.best_score_, 4))
    print("Best parameters:", search.best_params_)

    # Refit independently so that final full-training fit time is measurable.
    best_model = clone(search.best_estimator_)
    start = perf_counter()
    best_model.fit(X_train, y_train)
    fit_time = perf_counter() - start

    start = perf_counter()
    predicted = best_model.predict(X_val)
    predict_time = perf_counter() - start

    metrics = {
        "Model": model_name,
        "Best parameters": str(search.best_params_),
        "CV Negative F1": search.best_score_,
        "Validation Accuracy": accuracy_score(y_val, predicted),
        "Validation Negative F1": f1_score(y_val, predicted, pos_label=0, zero_division=0),
        "Validation Macro F1": f1_score(y_val, predicted, average="macro", zero_division=0),
        "CV search time (s)": round(search_time, 3),
        "Final training time (s)": round(fit_time, 3),
        "Validation prediction time (s)": round(predict_time, 3),
    }
    rows.append(metrics)
    print("Validation Negative F1:", round(metrics["Validation Negative F1"], 4))

    slug = (
        model_name.lower().replace(" + ", "_").replace(" ", "_")
    )
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(
        y_val, predicted,
        labels=[0, 1],
        display_labels=["Negative (0)", "Positive (1)"],
        cmap="Blues", values_format="d", colorbar=False, ax=ax,
    )
    ax.set_title(f"{model_name} — validation")
    fig.tight_layout()
    output_file = OUTPUT_DIR / f"{slug}_confusion_matrix.png"
    fig.savefig(output_file, dpi=150)
    plt.show()
    plt.close(fig)
    print("Saved:", output_file.name)

    # Display the selected hyperparameters.
    cv_summary = pd.DataFrame(search.cv_results_)[
        ["params", "mean_test_negative_f1", "mean_test_accuracy", "mean_test_macro_f1"]
    ].sort_values("mean_test_negative_f1", ascending=False)
    display(cv_summary.head(5).reset_index(drop=True))

## 6. Final model comparison (CSV)

This is the only CSV saved by the notebook. The **best model** is the one with the highest validation Negative F1 among the four configurations selected through training cross-validation. BERT and RNN results are comparable on the same validation split.

In [ ]:
comparison = pd.DataFrame(rows).sort_values(
    "Validation Negative F1", ascending=False
).reset_index(drop=True)

comparison.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)

display(comparison.round(4))
print("\nBest traditional ML method:", comparison.loc[0, "Model"])
print("\nSaved output files:")
for output_file in sorted(OUTPUT_DIR.iterdir()):
    if output_file.is_file():
        print(" -", output_file.name)

## Experiment summary

**Aim:** Find whether traditional ML classifiers can perform competitively with the RNN and BERT approaches, while requiring fewer computational resources.

**Procedure:** Keep the original train/validation split, tune each method with 3-fold stratified cross-validation (Negative F1), then evaluate its selected configuration once on validation. Record the best hyperparameters, validation Accuracy, Negative F1, Macro F1, training/prediction time, and confusion matrix.

**Note:** The test set has no labels, so it cannot be used to calculate test Accuracy or F1. Runtime comparisons against BERT/RNN should acknowledge hardware and measurement differences. The best parameters here are the best **among those tested**, not necessarily globally optimal.